# Supervised Fine-Tuning SFT Pipeline Demonstration

This notebook demonstrates the basic training pipeline of Supervised Fine-Tuning (SFT) over a toy language head to align outputs to specific formatting targets.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Set random seed for reproducibility
torch.manual_seed(42)

# Define a simple classifier representing our LLM vocabulary head
class ToyLanguageHead(nn.Module):
    def __init__(self, vocab_size=10, embed_dim=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)
        
    def forward(self, x):
        h = self.embedding(x)
        logits = self.linear(h)
        return logits

# Toy instruction SFT data: (prompts, responses)
# Map prompt IDs [1, 2, 3] to response target IDs [4, 5, 6]
prompts = torch.tensor([[1, 2, 3], [3, 2, 1]], dtype=torch.long)
targets = torch.tensor([[4, 5, 6], [6, 5, 4]], dtype=torch.long)

model = ToyLanguageHead()
optimizer = optim.AdamW(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

print("Initial Loss:")
logits = model(prompts)
# Shift for causal prediction
loss = criterion(logits[:, :-1, :].reshape(-1, 10), targets[:, 1:].reshape(-1))
print(f"Loss: {loss.item():.4f}")


Initial Loss:
Loss: 2.1537


### Output Explanation
The output log shows the initial loss of the randomly initialized network before SFT starts. The calculated cross-entropy evaluates the probability distribution matching of shifted tokens.

In [2]:
print("Training for 20 epochs...")
for epoch in range(20):
    optimizer.zero_grad()
    logits = model(prompts)
    loss = criterion(logits[:, :-1, :].reshape(-1, 10), targets[:, 1:].reshape(-1))
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | Loss: {loss.item():.4f}")


Training for 20 epochs...
Epoch 05 | Loss: 1.8150
Epoch 10 | Loss: 1.4611
Epoch 15 | Loss: 1.1720
Epoch 20 | Loss: 0.9354


### Output Explanation
The printed logs show the SFT loss steadily decreasing as the model parameters are updated to align with the instruction-following dataset target shifts.